# Step-Varying Cap Sweep — Random Token Rejection

**Goal.** Test whether a linearly-decaying cap schedule (high ρ early, low ρ late) beats the best static ρ at 8 and 16 steps.

**Intuition.** Early steps have a small KV cache and many tokens are predicted in parallel under high uncertainty — deferring more lets later steps benefit from richer context. Late steps have a large cache and tokens deferred there ultimately end up in the forced-accept final step, where there's no more cache growth to gain from. So defer aggressively early, conservatively late.

**Schedules tested (per step count):**

| Label | ρ start | ρ end | Note |
|---|---|---|---|
| `constant-0.7`   | 0.7 | 0.7 | Control = best static from previous sweep |
| `decay-mild`     | 0.8 | 0.5 | Mild aggressive→conservative |
| `decay-aggr`     | 0.9 | 0.3 | Aggressive decay |
| `decay-extreme`  | 0.95 | 0.2 | Pushing it |
| `reverse-grow`   | 0.3 | 0.7 | Sanity check — opposite direction |

**Configs.** 5 schedules × 2 step counts (8, 16) × 3 seeds = **30 FID-10K runs**, ~4 hours on A100.

**Persistence.** Master CSV at `MyDrive/ARPG-assets/results/final-paper/cap-schedule-random/results.csv`. Same resumability as the cap-sweep notebook: re-run end-to-end any time, completed configs are skipped.

**NPZ policy.** `KEEP_NPZ_ON_DRIVE = False` — only FID metrics + 8 qualitative PNGs + rejection JSON per config.

**Required.** The cloned fork must include the `max_reject_rate_end` parameter in `models/arpg.py` and the `--max-reject-rate-end` CLI flag in `sample_c2i_ddp.py`. Commit those to your fork before running this notebook (or run the hotfix cell).

## 1. Mount Drive and set up paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ARPG-assets')
RESULTS_ROOT = DRIVE_ROOT / 'results' / 'final-paper' / 'cap-schedule-random'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CSV_PATH = RESULTS_ROOT / 'results.csv'
SAMPLES_DIR = RESULTS_ROOT / 'samples'
SAMPLES_GRIDS_DIR = SAMPLES_DIR / 'grids'
SAMPLES_INDIVIDUAL_DIR = SAMPLES_DIR / 'individual'
SAMPLES_GRIDS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_INDIVIDUAL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = RESULTS_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
REJECTION_LOGS_DIR = RESULTS_ROOT / 'rejection-logs'
REJECTION_LOGS_DIR.mkdir(parents=True, exist_ok=True)

# NPZ persistence policy: do NOT keep large NPZs on Drive.
KEEP_NPZ_ON_DRIVE = False

REPO_LOCAL = Path('/content/ARPG-main')
LOCAL_SAMPLE_DIR = Path('/content/samples')
LOCAL_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

REF_NPZ = DRIVE_ROOT / 'eval' / 'VIRTUAL_imagenet256_labeled.npz'
ARPG_CKPT = DRIVE_ROOT / 'weights' / 'arpg_300m.pt'
VQ_CKPT = DRIVE_ROOT / 'weights' / 'vq_ds16_c2i.pt'
GD_REPO = DRIVE_ROOT / 'external' / 'guided-diffusion'

print(f'Results root: {RESULTS_ROOT}')
print(f'Master CSV  : {CSV_PATH}')
print(f'KEEP_NPZ_ON_DRIVE = {KEEP_NPZ_ON_DRIVE}')

## 2. Clone repo and verify random + step-varying support

In [ ]:
REPO_URL = 'https://github.com/rshahbazov23/comp447-arpg-private.git'
GITHUB_TOKEN = None  # e.g. 'ghp_...'

import subprocess, shutil

def _clone_url(url, token):
    if token and url.startswith('https://github.com/'):
        return url.replace('https://', f'https://{token}@')
    return url

if not REPO_LOCAL.exists():
    print(f'Cloning {REPO_URL} → {REPO_LOCAL}')
    subprocess.run(['git', 'clone', _clone_url(REPO_URL, GITHUB_TOKEN), str(REPO_LOCAL)], check=True)
else:
    print('Repo already present, pulling latest')
    subprocess.run(['git', '-C', str(REPO_LOCAL), 'pull'], check=True)

# Sanity-check required files
required = [
    (REF_NPZ,   'ImageNet reference batch'),
    (ARPG_CKPT, 'ARPG-L pretrained checkpoint'),
    (VQ_CKPT,   'LlamaGen VQ tokenizer'),
    (GD_REPO / 'evaluations' / 'evaluator.py', 'guided-diffusion evaluator'),
    (REPO_LOCAL / 'sample_c2i_ddp.py', 'ARPG sampling script'),
    (REPO_LOCAL / 'models' / 'arpg.py', 'ARPG model'),
    (REPO_LOCAL / 'models' / 'confidence.py', 'Confidence metrics module'),
]
for p, name in required:
    if not p.exists():
        raise FileNotFoundError(f'MISSING: {name} — expected at {p}')
print('All assets present.')

# Confirm random support + step-varying cap support
conf_src = (REPO_LOCAL / 'models' / 'confidence.py').read_text()
if 'random_score' not in conf_src:
    raise RuntimeError('confidence.py is missing random_score. Push latest from your local Mac.')

arpg_src = (REPO_LOCAL / 'models' / 'arpg.py').read_text()
if 'max_reject_rate_end' not in arpg_src:
    raise RuntimeError('arpg.py is missing the max_reject_rate_end parameter. '
                       'Push the step-varying cap commit from your local Mac.')

sample_src = (REPO_LOCAL / 'sample_c2i_ddp.py').read_text()
if '--max-reject-rate-end' not in sample_src:
    raise RuntimeError('sample_c2i_ddp.py is missing --max-reject-rate-end CLI flag. '
                       'Push the step-varying cap commit from your local Mac.')

print('Random + step-varying cap support confirmed.')

## 3. Install Python dependencies

In [ ]:
subprocess.run(['pip', 'install', '-q',
    'einops', 'transformers', 'scipy', 'tensorflow', 'pandas',
], check=True)

import torch, pandas as pd
print(f'torch  : {torch.__version__}, CUDA {torch.version.cuda}, GPU {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
assert torch.cuda.is_available(), 'No CUDA device — switch the Colab runtime to GPU.'

## 4. Config matrix and master CSV

Order: **8-step block runs first** (where the room is) so we get the most valuable data within one Colab session even if it disconnects. 16-step block runs after.

In [ ]:
import pandas as pd
from datetime import datetime

RTR_METRIC = 'random'
RTR_THRESHOLD = 2.0  # unreachable
NUM_FID_SAMPLES = 10_000

# Each schedule = (label, cap_start, cap_end). cap_end == cap_start is constant.
SCHEDULES = [
    ('constant-0.7',   0.7,  0.7),
    ('decay-mild',     0.8,  0.5),
    ('decay-aggr',     0.9,  0.3),
    ('decay-extreme',  0.95, 0.2),
    ('reverse-grow',   0.3,  0.7),
]

STEP_COUNTS_ORDER = [8, 16]
SEEDS = [0, 1, 2]

def make_configs():
    out = []
    for step in STEP_COUNTS_ORDER:
        for label, cap_start, cap_end in SCHEDULES:
            for seed in SEEDS:
                out.append({
                    'step': step,
                    'schedule': label,
                    'cap_start': cap_start,
                    'cap_end': cap_end,
                    'seed': seed,
                })
    return out

CONFIGS = make_configs()
print(f'Config matrix: {len(CONFIGS)} configs (2 steps × 5 schedules × 3 seeds)')

# Initialise master CSV if first run
if not CSV_PATH.exists():
    pd.DataFrame(columns=[
        'step', 'schedule', 'cap_start', 'cap_end', 'seed', 'fid',
        'inception_score', 'sfid', 'precision', 'recall',
        'npz_path', 'timestamp',
    ]).to_csv(CSV_PATH, index=False)
    print(f'Initialised empty master CSV: {CSV_PATH}')
else:
    df_master = pd.read_csv(CSV_PATH)
    print(f'Master CSV has {len(df_master)} existing rows.')

df_master = pd.read_csv(CSV_PATH)
done_keys = set(
    (int(r['step']), str(r['schedule']), int(r['seed']))
    for _, r in df_master.iterrows()
) if len(df_master) else set()
remaining = [c for c in CONFIGS
             if (c['step'], c['schedule'], c['seed']) not in done_keys]
print(f'\nProgress: {len(CONFIGS) - len(remaining)}/{len(CONFIGS)} done, {len(remaining)} remaining\n')
if remaining[:5]:
    print('First 5 configs to run:')
    for c in remaining[:5]:
        print(f'  step={c["step"]}  {c["schedule"]:<14}  cap={c["cap_start"]}->{c["cap_end"]}  seed={c["seed"]}')

## 5. Helper functions

In [ ]:
import re, time, traceback

QUALITATIVE_CLASSES = [207, 360, 388, 113, 355, 980, 323, 979]

def config_to_folder_name(cfg):
    base = (
        f'ARPG-L-arpg_300m-size-256-size-256-VQ-16-'
        f'topk-0-topp-1.0-temperature-1.0-cfg-5.0-cfg-schedule-linear-'
        f'sample-schedule-arccos-step-{cfg["step"]}-seed-{cfg["seed"]}-'
        f'mode-rejection-metric-{RTR_METRIC}-tau-{RTR_THRESHOLD}-cap-{cfg["cap_start"]}'
    )
    # Always include capend in the folder name (even when cap_start == cap_end)
    # so the schedule sweep configs don't collide with constant-cap sweep configs.
    base += f'-capend-{cfg["cap_end"]}'
    return base


def is_done(cfg, df_master):
    if not len(df_master):
        return False
    mask = (
        (df_master['step'].astype(int) == cfg['step'])
        & (df_master['schedule'].astype(str) == cfg['schedule'])
        & (df_master['seed'].astype(int) == cfg['seed'])
    )
    return bool(mask.any())


def cleanup_local_samples():
    if LOCAL_SAMPLE_DIR.exists():
        shutil.rmtree(LOCAL_SAMPLE_DIR)
    LOCAL_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)


def run_sampling(cfg, log_handle=None):
    folder = config_to_folder_name(cfg)
    cmd = [
        'torchrun', '--nnodes=1', '--nproc_per_node=1',
        'sample_c2i_ddp.py',
        '--gpt-model', 'ARPG-L',
        '--gpt-ckpt', str(ARPG_CKPT),
        '--vq-ckpt', str(VQ_CKPT),
        '--sample-schedule', 'arccos',
        '--cfg-schedule', 'linear',
        '--cfg-scale', '5.0',
        '--step', str(cfg['step']),
        '--per-proc-batch-size', '64',
        '--num-fid-samples', str(NUM_FID_SAMPLES),
        '--global-seed', str(cfg['seed']),
        '--sample-dir', str(LOCAL_SAMPLE_DIR),
        '--no-compile',
        '--precision', 'bf16',
        '--rejection-mode', 'rejection',
        '--confidence-metric', RTR_METRIC,
        '--rejection-threshold', str(RTR_THRESHOLD),
        '--max-reject-rate', str(cfg['cap_start']),
        '--max-reject-rate-end', str(cfg['cap_end']),
        '--log-json', str(REJECTION_LOGS_DIR / f'{folder}.json'),
    ]
    print(f'  Sampling: step={cfg["step"]} schedule={cfg["schedule"]} ({cfg["cap_start"]}->{cfg["cap_end"]}) seed={cfg["seed"]}')
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=str(REPO_LOCAL),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    last_print = t0
    for line in proc.stdout:
        if log_handle is not None:
            log_handle.write(line); log_handle.flush()
        now = time.time()
        if now - last_print > 60:
            print(f'    [{(now-t0)/60:.1f} min] {line.rstrip()[:120]}')
            last_print = now
    proc.wait()
    print(f'  Sampling done in {(time.time()-t0)/60:.1f} min (exit {proc.returncode})')
    if proc.returncode != 0:
        raise RuntimeError(f'Sampling failed for {cfg}: exit {proc.returncode}')
    local_npz = LOCAL_SAMPLE_DIR / f'{folder}.npz'
    if not local_npz.exists():
        raise FileNotFoundError(f'Expected NPZ not produced: {local_npz}')
    return local_npz


_METRIC_LINE = re.compile(r'^\s*(FID|sFID|Inception Score|Precision|Recall)\s*:\s*([0-9.eE+\-]+)')

def evaluate_fid(local_npz, log_handle=None):
    cmd = ['python', 'evaluations/evaluator.py', str(REF_NPZ), str(local_npz)]
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=str(GD_REPO), capture_output=True, text=True)
    print(f'  FID eval done in {(time.time()-t0)/60:.1f} min (exit {proc.returncode})')
    if log_handle is not None:
        log_handle.write('--- evaluator stdout ---\n')
        log_handle.write(proc.stdout)
        log_handle.write('\n--- evaluator stderr ---\n')
        log_handle.write(proc.stderr)
        log_handle.flush()
    if proc.returncode != 0:
        print('STDOUT (tail):', proc.stdout[-2000:])
        print('STDERR (tail):', proc.stderr[-2000:])
        raise RuntimeError(f'FID eval failed: exit {proc.returncode}')
    metrics = {}
    for line in proc.stdout.splitlines():
        m = _METRIC_LINE.match(line)
        if m:
            key = m.group(1).lower().replace(' ', '_')
            metrics[key] = float(m.group(2))
    if 'fid' not in metrics:
        print('STDOUT:', proc.stdout)
        raise ValueError('Could not parse FID from evaluator output')
    return metrics


def save_qualitative_samples(local_npz, cfg, num_classes=1000):
    import numpy as np
    from PIL import Image
    npz = np.load(str(local_npz))
    arr = npz['arr_0']
    folder = config_to_folder_name(cfg)
    indiv_dir = SAMPLES_INDIVIDUAL_DIR / folder
    indiv_dir.mkdir(parents=True, exist_ok=True)
    pils = []
    for cls in QUALITATIVE_CLASSES:
        idx = cls
        if idx >= arr.shape[0]:
            continue
        pil = Image.fromarray(arr[idx])
        pil.save(indiv_dir / f'class{cls:03d}.png')
        pils.append((cls, pil))
    if len(pils) == len(QUALITATIVE_CLASSES):
        w, h = pils[0][1].size
        ncols = 4
        nrows = (len(pils) + ncols - 1) // ncols
        grid = Image.new('RGB', (w * ncols, h * nrows), color=(255, 255, 255))
        for i, (_, pil) in enumerate(pils):
            r, c = divmod(i, ncols)
            grid.paste(pil, (c * w, r * h))
        grid.save(SAMPLES_GRIDS_DIR / f'{folder}_grid.png')
        print(f'  Saved {len(pils)} qualitative samples + grid')


def append_result(cfg, metrics, npz_path):
    df = pd.read_csv(CSV_PATH)
    row = {
        'step': cfg['step'],
        'schedule': cfg['schedule'],
        'cap_start': cfg['cap_start'],
        'cap_end': cfg['cap_end'],
        'seed': cfg['seed'],
        'fid': metrics.get('fid'),
        'inception_score': metrics.get('inception_score'),
        'sfid': metrics.get('sfid'),
        'precision': metrics.get('precision'),
        'recall': metrics.get('recall'),
        'npz_path': str(npz_path) if npz_path is not None else '(not kept)',
        'timestamp': datetime.now().isoformat(),
    }
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(CSV_PATH, index=False)
    return df


print('Helpers loaded.')

## 6. Main loop — resumable

Safe to interrupt at any time. Progress saved after each config.

In [ ]:
failures = []
started = datetime.now()

for i, cfg in enumerate(CONFIGS, 1):
    print(f'\n{"="*70}\n[{i}/{len(CONFIGS)}] step={cfg["step"]} schedule={cfg["schedule"]} ({cfg["cap_start"]}->{cfg["cap_end"]}) seed={cfg["seed"]}\n{"="*70}')

    df_master = pd.read_csv(CSV_PATH)
    if is_done(cfg, df_master):
        print('  SKIP (already in CSV)')
        continue

    folder = config_to_folder_name(cfg)
    local_npz = LOCAL_SAMPLE_DIR / f'{folder}.npz'
    log_path = LOG_DIR / f'{folder}.log'

    try:
        with open(log_path, 'w') as log_h:
            local_npz = run_sampling(cfg, log_handle=log_h)
            drive_npz = None  # NPZ stays on local disk only
            metrics = evaluate_fid(local_npz, log_handle=log_h)
            append_result(cfg, metrics, drive_npz)
            try:
                save_qualitative_samples(local_npz, cfg)
            except Exception as e:
                print(f'  Qualitative samples skipped: {e}')
        print(f'  DONE   FID={metrics["fid"]:.4f}')
    except Exception as e:
        print(f'  FAILED: {e}')
        traceback.print_exc()
        failures.append({'config': cfg, 'error': str(e), 'log': str(log_path)})
    finally:
        cleanup_local_samples()

    elapsed = (datetime.now() - started).total_seconds() / 3600
    print(f'  Session elapsed: {elapsed:.2f} h')

print(f'\n\n{"="*70}\nSESSION COMPLETE\n{"="*70}')
print(f'Configs attempted: {len(CONFIGS)}')
print(f'Failures: {len(failures)}')
for f in failures:
    print(f'  {f["config"]}  →  {f["error"]}  (log: {f["log"]})')

## 7. Summary — pick the best schedule per step count

Aggregates FID across seeds, compares each decay schedule to the constant ρ=0.7 control, and recommends a schedule for the §3 main table.

In [ ]:
import numpy as np

df = pd.read_csv(CSV_PATH)
if len(df) == 0:
    raise RuntimeError('Master CSV is empty — run the main loop first.')

df['step'] = df['step'].astype(int)
df['seed'] = df['seed'].astype(int)

print(f'Master CSV: {len(df)} rows\n')

summary = (
    df.groupby(['step', 'schedule'])['fid']
      .agg(['mean', 'std', 'count'])
      .round(4)
      .sort_index()
)
print('FID-10K — schedule comparison (mean ± std, n)\n')
print(summary.to_string())

# Per-step best schedule
print('\n\nBest schedule per step count:')
for step in sorted(df['step'].unique()):
    sub = df[df['step'] == step].groupby('schedule')['fid'].mean().sort_values()
    if len(sub) == 0:
        continue
    best = sub.index[0]
    best_fid = sub.iloc[0]
    # Compute deltas vs constant-0.7 control
    control = sub.get('constant-0.7', np.nan)
    print(f'\n  {step:>3} steps:   best = {best:<14}  FID-10K = {best_fid:.4f}')
    if not np.isnan(control):
        print(f'              control (constant-0.7) = {control:.4f}')
        for sched_name, fid in sub.items():
            delta = fid - control
            print(f'              {sched_name:<14}  {fid:.4f}   Δ = {delta:+.4f}')

# Visualise schedules + their FIDs
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, step in zip(axes, sorted(df['step'].unique())):
        sub = (df[df['step'] == step]
                 .groupby('schedule')['fid']
                 .agg(['mean', 'std', 'count'])
                 .reset_index())
        # Order schedules manually so the bar chart reads sensibly
        order = ['constant-0.7', 'decay-mild', 'decay-aggr', 'decay-extreme', 'reverse-grow']
        sub['order'] = sub['schedule'].map({s: i for i, s in enumerate(order)})
        sub = sub.dropna(subset=['order']).sort_values('order')
        ax.bar(sub['schedule'], sub['mean'], yerr=sub['std'], capsize=4)
        ax.set_xlabel('schedule')
        ax.set_ylabel('FID-10K (mean ± std, n=3)')
        ax.set_title(f'{step} decoding steps')
        for tick in ax.get_xticklabels():
            tick.set_rotation(15)
        ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    fig_path = RESULTS_ROOT / 'schedule_comparison.png'
    fig.savefig(fig_path, dpi=120)
    print(f'\nPlot saved to {fig_path}')
    plt.show()
except Exception as e:
    print(f'(Plotting skipped: {e})')

# Save summary CSV
summary_path = RESULTS_ROOT / 'summary.csv'
summary.to_csv(summary_path)
print(f'Summary CSV written to {summary_path}')